# Zava Model Router Fine-Tuning

This notebook walks through **fine-tuning the Azure Foundry Model Router** end-to-end using the bundled [`zava_enterprise`](../../Sample_Datasets/Model_Router_Fine_Tuning/zava_enterprise/) dataset (200 train / 100 test enterprise-operations prompts labelled for `gpt-5`, `gpt-5-mini`, and `gpt-5-nano` — chosen as a **representative subset** of the supported LLMs).

You will:

1. **Validate** the JSONL training and test files locally against the [Model Router schema](../../Sample_Datasets/Model_Router_Fine_Tuning/SCHEMA.md).
2. **Upload** the files to your Azure AI Foundry project.
3. **Submit** a fine-tuning job against the `model-router` base model.
4. **Monitor** the job to completion.
5. **Download** training metrics (`results.csv`).
6. **Deploy** the fine-tuned router via the Azure Management REST API.
7. **Test** the deployment with a sample prompt and see which underlying model was picked.

> 💡 **Heads-up — the 3-model subset is just an example.** Model Router fine-tuning supports labelling for **any subset of the [supported LLMs](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models)** (spanning GPT, Claude, Llama, DeepSeek, Grok, gpt-oss). The router you train will route between exactly the LLMs you label for.

> ⚠️ **Read [`README.md`](./README.md) first** for Model Router specific restrictions (the deployed router routes between the exact LLM set you labelled for; deploying requires the **Azure AI Owner** role).


## 1. Environment Setup

Install dependencies into the current kernel and load credentials from `.env`.

In [ ]:
# Safe to re-run — no-op if already installed
%pip install --quiet requests python-dotenv pandas tqdm

In [2]:
import json
import os
import time
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
# When opened from a different cwd, fall back to this file's directory if available
if not (NOTEBOOK_DIR / "README.md").exists():
    # Try the notebook's own directory if running via Jupyter
    try:
        NOTEBOOK_DIR = Path(__file__).parent  # type: ignore[name-defined]
    except NameError:
        pass

load_dotenv(NOTEBOOK_DIR / ".env")
print(f"Notebook dir: {NOTEBOOK_DIR}")

Notebook dir: <repo-root>\Demos\Zava_ModelRouter_FT


## 2. Configuration

All settings are read from environment variables — set them in a `.env` file at the demo root (copy from [`.env.template`](./.env.template)).

| Variable | Used by | Purpose |
|----------|---------|---------|
| `AZURE_OPENAI_PROJECT_ENDPOINT` | upload / submit / monitor / test | Azure AI Foundry **project** endpoint (e.g. `https://<resource>.services.ai.azure.com/api/projects/<project>`) |
| `AZURE_OPENAI_API_KEY` | upload / submit / monitor / test | API key for the project resource |
| `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZURE_RESOURCE_NAME` | deploy | Azure resource coordinates for the Management API |
| `AZURE_FINETUNED_DEPLOYMENT_NAME` | deploy / test | Name you choose for the new fine-tuned router deployment |
| `AZURE_AD_TOKEN` | deploy | AAD bearer token — run `az account get-access-token --query accessToken -o tsv` |


In [ ]:
# ── Data plane (upload / submit / monitor / test) ───────────────────────────
OPENAI_PROJECT_ENDPOINT = os.getenv("AZURE_OPENAI_PROJECT_ENDPOINT", "<your-project-endpoint>")
OPENAI_API_KEY          = os.getenv("AZURE_OPENAI_API_KEY",          "<your-azure-openai-api-key>")

# ── Control plane (deploy) ─────────────────────────────────────────────────
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "<your-subscription-id>")
RESOURCE_GROUP  = os.getenv("AZURE_RESOURCE_GROUP",  "<your-resource-group>")
RESOURCE_NAME   = os.getenv("AZURE_RESOURCE_NAME",   "<your-resource-name>")

FINETUNED_DEPLOYMENT_NAME   = os.getenv("AZURE_FINETUNED_DEPLOYMENT_NAME",   "zava-model-router-ft")
AZURE_AD_TOKEN              = os.getenv("AZURE_AD_TOKEN", "<your-azure-ad-bearer-token>")

# ── Data files ─────────────────────────────────────────────────────────────
DATA_DIR = (NOTEBOOK_DIR / ".." / ".." / "Sample_Datasets" / "Model_Router_Fine_Tuning" / "zava_enterprise").resolve()
TRAINING_FILE_PATH   = DATA_DIR / "zava_enterprise_train.jsonl"
VALIDATION_FILE_PATH = DATA_DIR / "zava_enterprise_test.jsonl"

# ── Fine-tuning job settings ───────────────────────────────────────────────
BASE_MODEL = "model-router"   # Model Router base — do NOT change this
SEED       = 105              # For reproducibility
JOB_API_VERSION = "v1"

# ── Deployment SKU ─────────────────────────────────────────────────────────
DEPLOYMENT_SKU      = "GlobalStandard"
DEPLOYMENT_CAPACITY = 10

# ── Inference test ─────────────────────────────────────────────────────────
INFERENCE_API_VERSION = "2025-04-01-preview"

# ── Polling ────────────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 60     # how often to check job status
FILE_PROCESSING_TIMEOUT = 120  # max seconds to wait for upload to finish processing

print("Training file:  ", TRAINING_FILE_PATH)
print("Validation file:", VALIDATION_FILE_PATH)
print("Base model:     ", BASE_MODEL)
print("Fine-tuned deployment:", FINETUNED_DEPLOYMENT_NAME)
assert TRAINING_FILE_PATH.exists(),   f"Training file not found: {TRAINING_FILE_PATH}"
assert VALIDATION_FILE_PATH.exists(), f"Validation file not found: {VALIDATION_FILE_PATH}"


## 3. Understanding the Data Format

Each line in the JSONL file is one prompt + per-model binary correctness `labels` + per-model `usage`. Both `labels` and `usage` are **required** on every row, and their key sets must match. See [`SCHEMA.md`](../../Sample_Datasets/Model_Router_Fine_Tuning/SCHEMA.md) for the full contract.

```json
{
  "messages": [{"role": "user", "content": "<your prompt>"}],
  "labels": {
    "gpt-5_2025-08-07":      1,
    "gpt-5-mini_2025-08-07": 1,
    "gpt-5-nano_2025-08-07": 0
  },
  "usage": {
    "gpt-5_2025-08-07":      {"prompt_tokens": 15, "completion_tokens": 1222},
    "gpt-5-mini_2025-08-07": {"prompt_tokens": 15, "completion_tokens": 702},
    "gpt-5-nano_2025-08-07": {"prompt_tokens": 15, "completion_tokens": 1349}
  }
}
```

`1` = the model answered correctly. The router learns to pick the **cheapest** model that's likely to be correct.


In [4]:
# Preview the first training example
with open(TRAINING_FILE_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("=== Sample Training Record ===")
print(json.dumps(sample, indent=2)[:2000])

=== Sample Training Record ===
{
  "messages": [
    {
      "role": "user",
      "content": "What's the current return rate for our cordless drill collection?"
    }
  ],
  "labels": {
    "gpt-5_2025-08-07": 1,
    "gpt-5-mini_2025-08-07": 1,
    "gpt-5-nano_2025-08-07": 0
  },
  "usage": {
    "gpt-5_2025-08-07": {
      "prompt_tokens": 15,
      "completion_tokens": 1222
    },
    "gpt-5-mini_2025-08-07": {
      "prompt_tokens": 15,
      "completion_tokens": 702
    },
    "gpt-5-nano_2025-08-07": {
      "prompt_tokens": 15,
      "completion_tokens": 1349
    }
  }
}


### Validate Training Data

Local check — no API calls. Runs the same rules as the import-time validator: required fields (`messages`, `labels`, `usage`), consistent model-key set across rows, binary `labels` values, and `usage` keys that match `labels`.

In [5]:
# NOTE: Model Router supports a curated set of LLM ids. We don't enumerate them
# here because the list grows over time — see the canonical catalog:
#   https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models
# The validator below checks structural well-formedness only (binary 0/1 labels,
# consistent key set across rows, 'usage' present with matching keys). The server
# will reject any unsupported LLM ids at job-submission time.

def validate_jsonl(path):
    errors, n, first_keys = [], 0, None
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            n += 1
            try:
                r = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f"Line {i}: invalid JSON — {e}"); continue
            if not isinstance(r.get("messages"), list) or not r["messages"]:
                errors.append(f"Line {i}: missing/empty 'messages'")
            else:
                for m in r["messages"]:
                    if not (isinstance(m, dict) and isinstance(m.get("role"), str) and isinstance(m.get("content"), str)):
                        errors.append(f"Line {i}: malformed message item"); break
            if not isinstance(r.get("labels"), dict) or not r["labels"]:
                errors.append(f"Line {i}: missing/empty 'labels'"); continue
            keys = set(r["labels"].keys())
            if first_keys is None:
                first_keys = keys
            elif keys != first_keys:
                errors.append(f"Line {i}: labels key set differs from first row "
                              f"(diff: {sorted(keys ^ first_keys)})")
            for k, v in r["labels"].items():
                if v not in (0, 1, "0", "1"):
                    errors.append(f"Line {i}: labels[{k!r}] must be 0/1, got {v!r}")
            # 'usage' is REQUIRED — must be a non-empty dict with the same key set as 'labels'
            if "usage" not in r:
                errors.append(f"Line {i}: missing required field 'usage'")
            else:
                u = r["usage"]
                if not isinstance(u, dict) or not u:
                    errors.append(f"Line {i}: 'usage' must be a non-empty dict")
                elif set(u.keys()) != keys:
                    errors.append(f"Line {i}: usage keys != labels keys")
                else:
                    for k, vv in u.items():
                        if not isinstance(vv, dict) or not isinstance(vv.get("prompt_tokens"), int):
                            errors.append(f"Line {i}: usage[{k!r}] missing integer 'prompt_tokens'")
    return n, sorted(first_keys or []), errors

for label, path in (("train", TRAINING_FILE_PATH), ("test", VALIDATION_FILE_PATH)):
    n, model_keys, errors = validate_jsonl(path)
    print(f"\n=== {label}: {path.name} ===")
    print(f"  rows:        {n}")
    print(f"  model keys:  {model_keys}")
    print(f"  errors:      {len(errors)}")
    for e in errors[:5]:
        print("   -", e)
    if errors:
        raise SystemExit("Fix validation errors before uploading.")


=== train: zava_enterprise_train.jsonl ===
  rows:        200
  model keys:  ['gpt-5-mini_2025-08-07', 'gpt-5-nano_2025-08-07', 'gpt-5_2025-08-07']
  errors:      0

=== test: zava_enterprise_test.jsonl ===
  rows:        100
  model keys:  ['gpt-5-mini_2025-08-07', 'gpt-5-nano_2025-08-07', 'gpt-5_2025-08-07']
  errors:      0


## 4. Upload Data

Upload the training (and optional validation) JSONL files to Azure Foundry via the REST API. We then wait until each file finishes server-side processing before submitting the fine-tuning job.

In [6]:
def upload_file(path):
    url = f"{OPENAI_PROJECT_ENDPOINT}/openai/v1/files"
    headers = {"api-key": OPENAI_API_KEY}
    with open(path, "rb") as f:
        resp = requests.post(
            url,
            headers=headers,
            files={"file": (Path(path).name, f, "application/octet-stream")},
            data={"purpose": "fine-tune"},
        )
    if not resp.ok:
        print(f"Upload failed ({resp.status_code}): {resp.text}")
    resp.raise_for_status()
    return resp.json()

def wait_for_file_processed(file_id, timeout=FILE_PROCESSING_TIMEOUT):
    url = f"{OPENAI_PROJECT_ENDPOINT}/openai/v1/files/{file_id}"
    headers = {"api-key": OPENAI_API_KEY}
    elapsed = 0
    while elapsed < timeout:
        resp = requests.get(url, headers=headers); resp.raise_for_status()
        status = resp.json().get("status", "")
        if status == "processed":
            return
        if status in {"error", "deleted"}:
            raise RuntimeError(f"File {file_id} ended in status: {status}")
        time.sleep(2); elapsed += 2
    print(f"Warning: file {file_id} still processing after {timeout}s")

print("Uploading training file...")
train_resp = upload_file(TRAINING_FILE_PATH)
training_file_id = train_resp["id"]
print(f"  Training file ID: {training_file_id}")
wait_for_file_processed(training_file_id)

validation_file_id = None
if VALIDATION_FILE_PATH and Path(VALIDATION_FILE_PATH).exists():
    print("Uploading validation file...")
    val_resp = upload_file(VALIDATION_FILE_PATH)
    validation_file_id = val_resp["id"]
    print(f"  Validation file ID: {validation_file_id}")
    wait_for_file_processed(validation_file_id)
else:
    print("No validation file — Azure will split training data 80:20 automatically.")


Uploading training file...
  Training file ID: file-da77e27322674e95b23db7a1399891bc
Uploading validation file...
  Validation file ID: file-f0fc0e33bdf64068b020b2a3f4854770


## 5. Submit the Fine-Tuning Job

Submit the job against the `model-router` base model.

> **Important:** the Model Router base requires the payload field `"trainingType": 1` (its dedicated training type). Omitting it, or sending `0`/`"Standard"`, returns `400 invalidPayload — does not support fine-tuning with Standard TrainingType`. The helper below includes this field; don't remove it.

In [7]:
def create_finetuning_job(training_file_id, validation_file_id=None, model=BASE_MODEL, seed=SEED):
    url = f"{OPENAI_PROJECT_ENDPOINT}/fine_tuning/jobs?api-version={JOB_API_VERSION}"
    headers = {"Content-Type": "application/json", "api-key": OPENAI_API_KEY}
    # NOTE: Model Router requires `trainingType: 1` (its dedicated non-Standard
    # training type). Omitting the field, or sending `0`/`Standard`, returns
    # `400 invalidPayload — does not support fine-tuning with Standard TrainingType`.
    payload = {"model": model, "training_file": training_file_id, "trainingType": 1}
    if validation_file_id:
        payload["validation_file"] = validation_file_id
    if seed is not None:
        payload["seed"] = seed
    resp = requests.post(url, headers=headers, json=payload)
    if not resp.ok:
        print(f"Job submission failed ({resp.status_code}): {resp.text}")
        print(f"Request URL:    {url}")
        print(f"Request payload: {json.dumps(payload, indent=2)}")
    resp.raise_for_status()
    return resp.json()

print("Submitting fine-tuning job...")
job_response = create_finetuning_job(training_file_id, validation_file_id)
job_id = job_response["id"]
print(f"Fine-tuning job submitted!")
print(f"  Job ID: {job_id}")
print(json.dumps(job_response, indent=2)[:1500])

Submitting fine-tuning job...
Fine-tuning job submitted!
  Job ID: ftjob-16faf28baa524a5591666873281a68b2
{
  "hyperparameters": {
    "n_epochs": 1,
    "batch_size": 16,
    "learning_rate_multiplier": 1
  },
  "additionalParameters": {
    "completionOverride": "True"
  },
  "seed": 105,
  "method": {
    "type": "supervised",
    "supervised": {
      "hyperparameters": {
        "n_epochs": 1,
        "batch_size": 16,
        "learning_rate_multiplier": 1
      }
    }
  },
  "trainingType": "globalStandard",
  "status": "pending",
  "model": "model-router-2025-11-18",
  "metadata": {
    "base_model": "model-router",
    "model_version": "2025-11-18"
  },
  "training_file": "file-da77e27322674e95b23db7a1399891bc",
  "validation_file": "file-f0fc0e33bdf64068b020b2a3f4854770",
  "estimated_finish": 1780036637,
  "id": "ftjob-16faf28baa524a5591666873281a68b2",
  "created_at": 1780034177,
  "object": "fine_tuning.job"
}


## 6. Monitor the Job

Poll until the job reaches a terminal state (`succeeded`, `failed`, or `cancelled`). This can take minutes to hours depending on dataset size and queue depth. Track progress in [Azure AI Foundry](https://ai.azure.com/) → **Fine-tuning** while this cell runs.

In [8]:
def get_job_status(job_id):
    url = f"{OPENAI_PROJECT_ENDPOINT}/fine_tuning/jobs/{job_id}?api-version={JOB_API_VERSION}"
    resp = requests.get(url, headers={"api-key": OPENAI_API_KEY})
    resp.raise_for_status()
    return resp.json()

def poll_until_complete(job_id, interval=POLL_INTERVAL_SECONDS):
    terminal = {"succeeded", "failed", "cancelled"}
    while True:
        status = get_job_status(job_id)
        current = status.get("status", "unknown")
        print(f"Job {job_id} status: {current}")
        if current in terminal:
            return status
        time.sleep(interval)

final_status = poll_until_complete(job_id)
print("\n=== Final Job Status ===")
print(json.dumps(final_status, indent=2)[:2000])

Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pending
Job ftjob-16faf28baa524a5591666873281a68b2 status: pendi

In [9]:
# Optional: list job events for troubleshooting
events_url = f"{OPENAI_PROJECT_ENDPOINT}/fine_tuning/jobs/{job_id}/events?api-version={JOB_API_VERSION}"
events_resp = requests.get(events_url, headers={"api-key": OPENAI_API_KEY})
events_resp.raise_for_status()
events = events_resp.json()
for ev in events.get("data", [])[:20]:
    print(f"[{ev.get('created_at')}] {ev.get('level')}: {ev.get('message')}")

[1780036171] info: Training tokens billed: 0
[1780036171] info: Completed results file: file-37ba4ceff5e14a809ad1aadc0eb9b60d
[1780036145] info: Job succeeded.
[1780035225] info: Step 0: FineTunedModelRouter_Quality=91
[1780035224] info: Finetuning started.
[1780035224] info: Created results file: file-37ba4ceff5e14a809ad1aadc0eb9b60d
[1780035187] info: Job started.
[1780035089] info: Preprocessing completed for file validation file.
[1780035088] info: Preprocessing running for file validation file.
[1780034584] info: Preprocessing completed for file training file.
[1780034223] info: Preprocessing running for file training file.
[1780034177] info: Job enqueued. Waiting for jobs ahead to complete.


## 7. Download Result File

On success, Azure Foundry generates a `results.csv` with per-step training metrics. Download and preview it.

In [10]:
if final_status.get("status") == "succeeded":
    result_files = final_status.get("result_files", [])
    if not result_files:
        print("No result files returned.")
    else:
        result_file_id = result_files[0]
        print(f"Result file ID: {result_file_id}")
        url = f"{OPENAI_PROJECT_ENDPOINT}/openai/v1/files/{result_file_id}/content"
        resp = requests.get(url, headers={"api-key": OPENAI_API_KEY})
        resp.raise_for_status()
        output_path = NOTEBOOK_DIR / "results.csv"
        with open(output_path, "wb") as f:
            f.write(resp.content)
        print(f"Saved to: {output_path}")
        import pandas as pd
        df = pd.read_csv(output_path)
        print(df.head(10))
else:
    print(f"Job did not succeed. Status: {final_status.get('status')}")
    print("Check the events list above for error details.")

Result file ID: file-37ba4ceff5e14a809ad1aadc0eb9b60d
Saved to: <repo-root>\Demos\Zava_ModelRouter_FT\results.csv
   step  BaseModelRouter_Cost  BaseModelRouter_Quality  \
0     0               0.56181                       89   

   FineTunedModelRouter_Cost  FineTunedModelRouter_Quality  \
0                   0.334333                            91   

   CostliestModel_Cost  CostliestModel_Quality  
0             3.724955                      91  


## 8. Deploy the Fine-Tuned Router

Deployment uses the **Azure Management REST API** (control plane), which requires an **Azure AD bearer token** — not the API key. Generate one with:

```bash
az login
az account get-access-token --query accessToken -o tsv
```

> You need the **Azure AI Owner** role on the resource to create deployments.

> ⚠️ **The deployed router routes between the exact LLMs you labelled for.** In this notebook that's `gpt-5`, `gpt-5-mini`, and `gpt-5-nano` — chosen as a representative subset. To target a different set, build a JSONL labelled for those LLMs (see the canonical [supported LLMs](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models) list) and re-run the notebook. You can deploy the resulting router in any [Model Router mode](https://learn.microsoft.com/azure/ai-services/openai/how-to/model-router) supported by your subscription.

In [ ]:
fine_tuned_model = final_status.get("fine_tuned_model")
print(f"Fine-tuned model: {fine_tuned_model}")
assert fine_tuned_model, "No fine-tuned model name on the job — ensure the job succeeded."

def deploy_finetuned_router(token, fine_tuned_model, deployment_name,
                            sku=DEPLOYMENT_SKU, capacity=DEPLOYMENT_CAPACITY):
    url = (
        f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
        f"/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}"
        f"/deployments/{deployment_name}?api-version=2025-06-01"
    )
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    payload = {
        "sku": {"name": sku, "capacity": capacity},
        "properties": {
            "model": {
                "format": "OpenAI",
                "name": fine_tuned_model,
                "version": "1",
            }
        },
    }
    resp = requests.put(url, headers=headers, json=payload)
    if not resp.ok:
        print(f"Deployment failed ({resp.status_code}): {resp.text}")
    resp.raise_for_status()
    print("Deployment created!")
    print(json.dumps(resp.json(), indent=2)[:1500])

deploy_finetuned_router(AZURE_AD_TOKEN, fine_tuned_model, FINETUNED_DEPLOYMENT_NAME)

Fine-tuned model: model-router-2025-11-18.ft-16faf28baa524a5591666873281a68b2
<REDACTED_AZURE_AD_TOKEN>
Deployment created!
{
  "id": "/subscriptions/<SUBSCRIPTION_ID>/resourceGroups/<RESOURCE_GROUP>/providers/Microsoft.CognitiveServices/accounts/<RESOURCE_NAME>/deployments/zava-model-router-ft",
  "type": "Microsoft.CognitiveServices/accounts/deployments",
  "name": "zava-model-router-ft",
  "sku": {
    "name": "GlobalStandard",
    "capacity": 10
  },
  "properties": {
    "model": {
      "format": "OpenAI",
      "name": "model-router-2025-11-18.ft-16faf28baa524a5591666873281a68b2",
      "version": "1"
    },
    "versionUpgradeOption": "NoAutoUpgrade",
    "currentCapacity": 10,
    "capabilities": {
      "router": "true",
      "chatCompletion": "true",
      "agentsV2": "true"
    },
    "raiPolicyName": "Microsoft.DefaultV2",
    "provisioningState": "Creating",
    "rateLimits": [
      {
        "key": "request",
        "renewalPeriod": 60,
        "count": 10
      },
  

## 9. Test the Deployed Router

Send a single prompt to the fine-tuned deployment and print (a) which underlying model the router picked and (b) a short preview of the response. The `model` field in the response is the routing decision — that's what fine-tuning was designed to change.

In [32]:
# Resolve the inference endpoint (strip the /api/projects/... path from the project endpoint)
_parsed = urlparse(OPENAI_PROJECT_ENDPOINT)
INFERENCE_ENDPOINT = f"{_parsed.scheme}://{_parsed.netloc}"

test_messages = [
    {"role": "user", "content": "Write a brief, friendly Slack announcement (3-4 sentences) letting the team know that the weekly engineering sync has moved from Monday 10am to Tuesday 2pm starting next week."}
]

print("=== Test Prompt ===")
print(test_messages[0]["content"])
print()

url = f"{INFERENCE_ENDPOINT}/openai/deployments/{FINETUNED_DEPLOYMENT_NAME}/chat/completions?api-version={INFERENCE_API_VERSION}"
headers = {"Content-Type": "application/json", "api-key": OPENAI_API_KEY}
resp = requests.post(url, headers=headers, json={"messages": test_messages})
resp.raise_for_status()
result = resp.json()

picked_model = result.get("model")
reply = result["choices"][0]["message"]["content"]
preview = reply[:300] + ("…" if len(reply) > 300 else "")

print(f"Fine-tuned router ({FINETUNED_DEPLOYMENT_NAME})  →  picked `{picked_model}`")
print()
print("=== Response (first 300 chars) ===")
print(preview)

=== Test Prompt ===
Write a brief, friendly Slack announcement (3-4 sentences) letting the team know that the weekly engineering sync has moved from Monday 10am to Tuesday 2pm starting next week.

Fine-tuned router (zava-model-router-ft)  →  picked `gpt-5-mini-2025-08-07`

=== Response (first 300 chars) ===
Quick update: starting next week our weekly engineering sync is moving from Monday at 10:00 AM to Tuesday at 2:00 PM. I’ve updated the calendar invite — please accept the new time if you haven’t already. If that creates a conflict, ping me and we’ll work it out.


## Next Steps

- **Bring your own prompts** — replace the Zava dataset with your own enterprise prompts. Label them against the LLM subset you want to route between — see the canonical [supported LLMs](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router#supported-models) list on Microsoft Learn. The GPT-5 trio here is just a representative subset.
- **Evaluate** — compare your fine-tuned router against the stock `model-router` deployment on a held-out test set to measure accuracy + cost improvement.

See the [Microsoft Foundry Model Router docs](https://learn.microsoft.com/azure/ai-services/openai/how-to/model-router) for runtime concepts, pricing, and the full list of supported models.
